In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
import warnings
import streamlit as st
import glob
import os

# Code GPT 5.1 11-21-25
def rebase_time_gaps(df, time_col='time', gap_threshold=2.0, reset_gap=0.01):
    """
    Rebases timestamps when gaps exceed a threshold.
    
    Parameters:
        df : pd.DataFrame
            Must contain a numeric or datetime time column.
        time_col : str
            Name of the time column.
        gap_threshold : float
            Threshold (in seconds) for gap detection.
        reset_gap : float
            Gap to insert after rebasing (in seconds).
    Returns:
        pd.DataFrame with continuous time column.
    """
    df = df.copy()
    
    # Convert to numeric seconds if datetime
    if not pd.api.types.is_numeric_dtype(df[time_col]):
        df[time_col] = pd.to_datetime(df[time_col])
        df[time_col] = (df[time_col] - df[time_col].iloc[0]).dt.total_seconds()
    
    # Sort by time
    df = df.sort_values(time_col).reset_index(drop=True)
    
    # Detect large gaps
    time_vals = df[time_col].to_numpy()
    gaps = np.diff(time_vals)
    
    # Keep track of total offset applied
    offset = 0.0
    adjusted_times = [time_vals[0]]
    
    for i, gap in enumerate(gaps, start=1):
        if gap > gap_threshold:
            # Increase offset by the size of the gap minus desired reset gap
            offset += (gap - reset_gap)
        adjusted_times.append(time_vals[i] - offset)
    
    df[time_col] = adjusted_times
    return df


def aursad_data():
    data_path = "../data/aursad"

    # Get all feather files, sorted in order (important for time series)
    feather_files = sorted(glob.glob(os.path.join(data_path, "part_*.feather")))

    # Load and concatenate
    start = 2
    stop = 6
    df_aursad = pd.concat([pd.read_feather(f) for f in feather_files[start:stop]], ignore_index=True)

    df_aursad = df_aursad.rename(columns={'timestamp': 'time'})
    df_aursad['time'] = df_aursad['time'] - df_aursad['time'].min()
    df_aursad = df_aursad.sort_values('time').reset_index(drop=True)
    df_aursad = rebase_time_gaps(df_aursad, time_col='time', gap_threshold=2.0, reset_gap=0.01)

    # Downsample
    df_aursad = df_aursad.iloc[::50]

    # Renaming to match CobotOps
    for i in range(6):
        df_aursad = df_aursad.rename(columns={f'actual_current_{i}': f'Current_J{i}'})
        df_aursad = df_aursad.rename(columns={f'actual_TCP_speed_{i}': f'Speed_J{i}'})
        df_aursad = df_aursad.rename(columns={f'joint_temperatures_{i}': f'Temperature_J{i}'})

    # Encode labels for screwing failures
    df_aursad = pd.get_dummies(df_aursad, columns=['label'], prefix='label')
    label_names = ["Normal operation", "Damaged screw", "Extra assembly component", "Missing screw", "Damaged thread samples", "Screw Loosening"]

    for i, label in enumerate(label_names):
        df_aursad = df_aursad.rename(columns={f'label_{i}': label})
    df_aursad.head()

    return df_aursad

df = aursad_data()


In [15]:
df.head()
with pd.option_context('display.max_rows', None):
    print(df.iloc[1])

sample_nr                            278
time                                 0.5
target_q_0                      0.086743
target_q_1                     -1.085944
target_q_2                       1.30411
target_q_3                     -0.174707
target_q_4                     -0.037001
target_q_5                     -1.605179
target_qd_0                          0.0
target_qd_1                          0.0
target_qd_2                          0.0
target_qd_3                          0.0
target_qd_4                          0.0
target_qd_5                          0.0
target_qdd_0                         0.0
target_qdd_1                         0.0
target_qdd_2                         0.0
target_qdd_3                         0.0
target_qdd_4                         0.0
target_qdd_5                         0.0
target_current_0                    -0.0
target_current_1               -1.143107
target_current_2               -0.777337
target_current_3                0.015464
target_current_4

In [9]:
df_angles = df[['actual_q_0', 'actual_q_1', 'actual_q_2', 'actual_q_3', 'actual_q_4', 'actual_q_5']]
df_angles

,actual_q_0,actual_q_1,actual_q_2,actual_q_3,actual_q_4,actual_q_5
0,0.086743,-1.085992,1.304120,-0.174706,-0.036994,-1.605189
50,0.086768,-1.085932,1.304108,-0.174695,-0.036994,-1.605177
100,0.086755,-1.085956,1.304108,-0.174659,-0.036994,-1.605189
150,0.086755,-1.085992,1.304120,-0.174719,-0.037018,-1.605177
200,0.086701,-1.085956,1.304120,-0.174695,-0.037031,-1.605165
...,...,...,...,...,...,...
333050,-0.063772,-1.758364,-0.067981,1.846091,0.432136,-1.586837
333100,0.001058,-1.468695,0.522843,0.975995,0.229896,-1.594739
333150,0.048193,-1.258177,0.952855,0.342473,0.083099,-1.600480
333200,0.076729,-1.130371,1.213191,-0.041151,-0.005915,-1.603967
